# Phase 2 — baselines + 5-way multi-pretrained-init bake-off

**Reference:** master plan v5.5 (Phase 2 section), `master_checklist.txt` Block C (C.1–C.7).

**Hardware:** RunPod RTX 4090 (Community), ~$0.34–0.50/hr. Total cost expectation for this notebook: ~$10 GPU, ~10 hr wall-clock.

**What you'll get out of this notebook:**

1. **Educational walkthrough** of the Phase 2 strategy (read on your laptop in ~15 min, no execution needed).
2. **Runnable code** to execute every Phase 2 experiment (open the same notebook on the pod via Jupyter, run cells in order).
3. **Results summary template** to fill in after each Kaggle submission.

**Two reading modes:**

- *Reading on Windows laptop*: just read the markdown + skim the code cells. Don't run anything that touches GPU or ultralytics.
- *Running on RunPod pod*: after C.1 deploys the pod, `git clone` + `pip install -r requirements.txt`, then open this notebook in Jupyter and execute Sections 1+2 once for setup, then Sections 3-6 to run experiments.

---

## Why this notebook exists (vs. just running the scripts)

The 4 production scripts (`download_marine_models.py`, `train_phase2_baseline.py`, `eval_pretrained_inits.py`, `predict_test_set.py`) handle all the actual computation. This notebook exists for three reasons:

1. **Pedagogy.** It explains the *why* behind each step. Comments in scripts get terse fast; markdown in a notebook can breathe.
2. **Working notes paper material.** The CLEF working notes paper (May 28 deadline) needs a clean methodology section. A notebook with narrative + code + final outputs is exactly the source material a paper section is built from.
3. **Recovery / resume.** If a step fails halfway, you can re-run individual cells without re-running the whole pipeline. Scripts are batch; notebooks are interactive.

Use the scripts for the bulk training jobs (long-running, walk-away). Use this notebook for setup, exploration, and result-recording.

# Section 1 — Phase 2 strategy, in plain English

## The competition reality

FathomNet 2026 is a deep-sea object detection task: 32 marine biology classes, 6,463 train images, 1,425 test images. The metric is COCO mAP@[.50:.95] (averaged across 10 IoU thresholds, weighted equally per class).

Phase 0 EDA confirmed three pain points:

| Pain point | Magnitude | Why it matters |
|---|---|---|
| Class imbalance | 817:1 (urchin: 5,723 vs sea slug: 7) | Frequent classes dominate gradient signal; rare classes barely train |
| Single-category-image rate | 62.5% of train images contain only ONE labeled class | Other classes present in those images are unlabeled — Positive-Unlabeled (PU) learning |
| Test resolution shift | 14% of test images are 720×486 vs. train at 1920×1080+ | Inference at training imgsz under-detects on the low-res subset |

The metric averages each class equally, so missing the 7 sea slug instances costs as much per-class as missing 5,723 urchins. **Rare classes are the win condition.**

## Why the choice of pretrained init matters more than people think

When you fine-tune a detector, you're starting from some checkpoint and nudging weights toward the target dataset. The closer the starting weights are to the target, the less nudging is needed → higher final mAP for the same training time + budget.

Most teams will start from `yolo11m.pt` (COCO weights — 80 generic land/web classes). COCO is genuinely the WORST possible starting domain for FathomNet:

- COCO class examples: person, car, kitchen objects, animals
- FathomNet class examples: amphipod, sea pen, calycophoran siphonophore, deep-sea sponge

Visual statistics barely overlap: lighting (sunlit terrestrial vs. deep-sea blue-green darkness), color, contrast, object shapes, image resolution, even aspect ratios. Domain gap is enormous.

Per the v5.5 strategic pivot (driven by host Laura Chrobak's Rule 6 clarification), we now know that publicly-available marine-specific pretrained detectors are explicitly allowed. There are three of them, and two were trained on FathomNet imagery directly:

| Model | Architecture | Pretraining | Domain match |
|---|---|---|---|
| **YOLOv11m + COCO** | YOLOv11m | MS-COCO 2017 (80 classes) | Worst |
| **YOLOv11m + BioCLIP2** | YOLOv11m + ViT-L/14 adapter | TreeOfLife-10M biology images | Medium (biology, not marine) |
| **YOLOv8x + Megalodon** | YOLOv8x | ALL FathomNet localizations (single "object" class) | Strong (right distribution, but only objectness) |
| **YOLOv8 + MBARI 315k** | YOLOv8 | 315k MBARI deep-sea benthic, multi-class | **STRONGEST** (right distribution AND multi-class) |
| **YOLOv5m + Megafishdetector** | YOLOv5m | Multi-source fish (incl. FathomNet subset) | Strong for fish classes only |

## Why we run a 5-way bake-off instead of guessing

Without testing, we'd have to guess which init is best. Guessing wrong could cost more than ANY single Phase 4-7 differentiator.

The bake-off solves this empirically: train all 5 candidates identically for 10 epochs on fold 0, rank by val mAP@[.50:.95], take the top 2 to full training, use both as ensemble members later.

Cost: ~$10 GPU, ~10 hr wall-clock. Information gained: which init the actual data prefers, plus 2 architecturally diverse models for the Phase 7 weighted-box-fusion ensemble.

Most teams won't do this. That's the differentiator.

## Decision rules (codified, will be applied automatically by the bake-off script)

- Take the **top 2** by val mAP at the screening epoch count.
- If `top_1.val_mAP < 0.15`: pipeline bug. STOP and debug (don't blame the inits).
- If `top_1 - top_2 gap < 0.02`: also keep `top_3` for ensemble (close race → more diversity).
- If `top_1 - top_2 gap > 0.05`: drop `top_2` (one model dominates → don't dilute it).

# Section 2 — Pre-flight checks (run on the pod)

Cells in this section verify everything the Phase 2 experiments need is in place: dataset, fold YAMLs, BioCLIP2 weights, marine model weights, W&B login. Run them once after deploying the pod.

If any cell fails, fix the underlying issue (usually missing weights) before proceeding to Section 3.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.getcwd())
while not (REPO_ROOT / "src" / "submit.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src" / "submit.py").exists(), "Could not find repo root from cwd; cd into the repo first"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"Python:    {sys.executable}")
print(f"Platform:  {sys.platform}")

In [ ]:
"""Verify the dataset is in place: train/test images, COCO JSONs, fold YAMLs."""
checks = []

train_imgs_dir = REPO_ROOT / "data" / "raw" / "images" / "train"
test_imgs_dir  = REPO_ROOT / "data" / "raw" / "images" / "test"
train_json     = REPO_ROOT / "data" / "raw" / "train_dataset.json"
test_json      = REPO_ROOT / "data" / "raw" / "test_dataset.json"
labels_dir     = REPO_ROOT / "data" / "raw" / "labels" / "train"
folds_dir      = REPO_ROOT / "data" / "folds"

n_train_imgs = sum(1 for _ in train_imgs_dir.glob("*")) if train_imgs_dir.exists() else 0
n_test_imgs  = sum(1 for _ in test_imgs_dir.glob("*"))  if test_imgs_dir.exists()  else 0
n_label_txt  = sum(1 for _ in labels_dir.glob("*.txt")) if labels_dir.exists()     else 0
n_fold_yaml  = sum(1 for _ in folds_dir.glob("fold*.yaml")) if folds_dir.exists()  else 0

checks.append(("train_dataset.json", train_json.exists(), str(train_json)))
checks.append(("test_dataset.json",  test_json.exists(),  str(test_json)))
checks.append(("train images", n_train_imgs >= 6000, f"{n_train_imgs:,} found (expected ~6,463)"))
checks.append(("test images",  n_test_imgs >= 1400,  f"{n_test_imgs:,} found (expected ~1,425)"))
checks.append(("YOLO labels",  n_label_txt >= 6000,  f"{n_label_txt:,} .txt files (run scripts/coco_to_yolo.py if missing)"))
checks.append(("fold YAMLs",   n_fold_yaml == 5,     f"{n_fold_yaml} found (expected 5; run scripts/make_yolo_fold.py if missing)"))

for name, ok, info in checks:
    sym = "OK  " if ok else "FAIL"
    print(f"[{sym}] {name:<22} -- {info}")

if not all(ok for _, ok, _ in checks):
    print("\n!! Fix failures above before proceeding to Section 3.")

In [ ]:
"""Verify pretrained weights are downloaded.

Run scripts/download_marine_models.py on the pod if any marine model is missing.
BioCLIP2 is pulled by `snapshot_download('imageomics/bioclip-2', ...)` per D.1.
"""
weights_dir = REPO_ROOT / "weights"
bioclip2    = weights_dir / "bioclip2"
megalodon   = weights_dir / "marine_models" / "megalodon"
mbari_315k  = weights_dir / "marine_models" / "mbari_315k"
megafish    = weights_dir / "marine_models" / "megafishdetector"

def _summarize(d: Path):
    if not d.exists():
        return False, "directory missing"
    pts = list(d.rglob("*.pt"))
    safetensors = list(d.rglob("*.safetensors"))
    bins = list(d.rglob("*.bin"))
    artifacts = pts + safetensors + bins
    if not artifacts:
        return False, "no .pt/.safetensors/.bin found"
    total_mb = sum(p.stat().st_size for p in artifacts) / 1024 / 1024
    return True, f"{len(artifacts)} weight file(s), {total_mb:.0f} MB total"

for name, path in [
    ("BioCLIP2 (D.1)",   bioclip2),
    ("Megalodon (D.7)",  megalodon),
    ("MBARI 315k (D.8)", mbari_315k),
    ("Megafishdetector (D.9)", megafish),
]:
    ok, info = _summarize(path)
    sym = "OK  " if ok else "MISSING"
    print(f"[{sym}] {name:<24} -- {info}")

In [ ]:
"""Verify Ultralytics can load fold 0 (catches FLAG 1 cat_id mapping bugs early)."""
fold0_yaml = folds_dir / "fold0.yaml"
if not fold0_yaml.exists():
    print(f"FAIL  fold0.yaml not found at {fold0_yaml}; run scripts/make_yolo_fold.py first")
else:
    try:
        from ultralytics.data.utils import check_det_dataset
        report = check_det_dataset(str(fold0_yaml))
        n_train = len(report.get("train", []))   if isinstance(report.get("train"), list)   else "?"
        n_val   = len(report.get("val", []))     if isinstance(report.get("val"), list)     else "?"
        nc      = report.get("nc", "?")
        names   = report.get("names", {})
        print(f"OK    fold0.yaml loaded: nc={nc}, n_train={n_train}, n_val={n_val}")
        print(f"      class 0:  {names.get(0)}")
        print(f"      class 31: {names.get(31)}")
    except Exception as exc:
        print(f"FAIL  Ultralytics check_det_dataset: {exc.__class__.__name__}: {exc}")

In [ ]:
"""Verify W&B login (per W1-W6 protocol in master_checklist.txt)."""
import os
api_key_present = bool(os.environ.get("WANDB_API_KEY"))
print(f"WANDB_API_KEY env var present: {api_key_present}")

try:
    import wandb
    print(f"wandb version: {wandb.__version__}")
    api = wandb.Api()
    runs = api.runs("navya2392-usc/fathomnet-2026", per_page=1)
    sample = next(iter(runs), None)
    print(f"OK  -- W&B project accessible (sample run: {sample.id if sample else 'none yet'})")
except Exception as exc:
    print(f"FAIL  W&B check: {exc.__class__.__name__}: {exc}")
    print("      Run `wandb login` and ensure WANDB_API_KEY is set.")

# Section 3 — EXP 2.1: Nano sanity check (~10 min, $0.10)

**Goal:** verify the entire pipeline runs end-to-end on real data. Not a serious training run.

**What we're checking:**
- Ultralytics can load fold 0 successfully
- Training loop completes 10 epochs without crashing
- val mAP@[.50:.95] lands in the 0.05-0.12 range (sanity threshold)
- W&B run logs land in the dashboard

**If this fails:** Section 4 will too. Fix here first.

**Cost:** ~10 min on RTX 4090, costs about $0.08.

In [ ]:
"""EXP 2.1 -- nano sanity check.

Equivalent CLI:
    python scripts/train_phase2_baseline.py --init yolo11n.pt --epochs 10 \
        --imgsz 640 --fold 0 --name p2_smoke_yolo11n
"""
from scripts.train_phase2_baseline import TrainConfig, train_one_run

smoke_cfg = TrainConfig(
    init="yolo11n.pt",
    epochs=10,
    imgsz=640,
    fold=0,
    batch=16,
    name="p2_smoke_yolo11n",
    extra_tags=["smoke"],
)
smoke_results = train_one_run(smoke_cfg)
print()
print("=== EXP 2.1 nano sanity result ===")
final = smoke_results.get("final_metrics", {})
print(f"  val mAP50:      {final.get('metrics/mAP50(B)', 'unknown')}")
print(f"  val mAP50-95:   {final.get('metrics/mAP50-95(B)', 'unknown')}")
print(f"  best.pt:        {smoke_results.get('best_pt')}")
print(f"  wall time:      {smoke_results.get('wall_time_sec', 0)/60:.1f} min")

# Section 4 — EXP 2.2: Vanilla baseline + first Kaggle submission (~2.5 hr, ~$1.5)

**Goal:** train YOLOv11m + COCO weights for 50 epochs on fold 0, submit to Kaggle as the FIRST anchor (per first-submission anchoring rule).

**Why first-submission must be vanilla:**
1. Validates the entire submission pipeline (predict -> CSV -> validate -> upload)
2. Anchors the leaderboard so every later experiment can claim "+X mAP over our anchor"
3. Burns 1 of 10 daily submission slots productively (the slot was going to be wasted on a sanity check anyway)

**Expected val mAP@[.50:.95]:** 0.18-0.28 after 50 epochs. The leaderboard score is typically ~0.05-0.10 below the val score (test set is harder).

**This becomes Init Candidate #1 in the Section 5 bake-off.**

In [ ]:
"""EXP 2.2 -- vanilla baseline.

Equivalent CLI:
    python scripts/train_phase2_baseline.py --init yolo11m.pt --epochs 50 \
        --imgsz 640 --fold 0 --name p2_baseline_yolo11m_coco
"""
baseline_cfg = TrainConfig(
    init="yolo11m.pt",
    epochs=50,
    imgsz=640,
    fold=0,
    batch=16,
    name="p2_baseline_yolo11m_coco",
    extra_tags=["baseline", "anchor"],
)
baseline_results = train_one_run(baseline_cfg)
print()
print("=== EXP 2.2 vanilla baseline result ===")
final = baseline_results.get("final_metrics", {})
print(f"  val mAP50:      {final.get('metrics/mAP50(B)', 'unknown')}")
print(f"  val mAP50-95:   {final.get('metrics/mAP50-95(B)', 'unknown')}")
print(f"  best.pt:        {baseline_results.get('best_pt')}")
print(f"  wall time:      {baseline_results.get('wall_time_sec', 0)/60:.1f} min")

In [ ]:
"""Predict on test set and write Kaggle CSV.

Equivalent CLI:
    python scripts/predict_test_set.py \
        --weights weights/runs/p2_baseline_yolo11m_coco/weights/best.pt \
        --out submissions/p2_baseline.csv
"""
import subprocess
weights_path = baseline_results["best_pt"]
out_csv = REPO_ROOT / "submissions" / "p2_baseline.csv"
out_csv.parent.mkdir(exist_ok=True)

cmd = [
    sys.executable, str(REPO_ROOT / "scripts" / "predict_test_set.py"),
    "--weights", str(weights_path),
    "--out", str(out_csv),
    "--imgsz", "640",
]
print(" ".join(cmd))
subprocess.check_call(cmd, cwd=str(REPO_ROOT))
print(f"\nReady to submit: {out_csv}")
print(f"Submit cmd:")
print(f"  kaggle competitions submit -c fathomnet-2026 -f {out_csv} \\")
print(f"    -m 'Phase 2 vanilla yolo11m+COCO baseline (anchoring)'")

# Section 5 — EXP 2.3: 5-way multi-pretrained-init bake-off (~5 hr, ~$3) — THE differentiator

**Goal:** train all 5 candidate inits identically for 10 epochs on fold 0, rank by val mAP@[.50:.95], pick top 2.

**Why this matters:** see Section 1 — most teams won't try multi-init; we likely gain 0.05-0.10 mAP for free.

**This is a long-running cell (~5 hr).** Start it, walk away, come back when it's done. The script writes a ranked Markdown table to `notes/phase2_init_bakeoff.md` at the end.

**Logging:** every candidate's run is tagged with `["phase2", "bakeoff", <init_name>]` in W&B for filterable comparison.

In [ ]:
"""EXP 2.3 -- 5-way multi-pretrained-init bake-off.

Equivalent CLI:
    python scripts/eval_pretrained_inits.py --epochs 10 --fold 0 --imgsz 640
"""
cmd = [
    sys.executable, str(REPO_ROOT / "scripts" / "eval_pretrained_inits.py"),
    "--epochs", "10",
    "--fold", "0",
    "--imgsz", "640",
    "--batch", "16",
]
print(" ".join(cmd))
subprocess.check_call(cmd, cwd=str(REPO_ROOT))

In [ ]:
"""Inspect the bake-off ranking + decision."""
bakeoff_md = REPO_ROOT / "notes" / "phase2_init_bakeoff.md"
bakeoff_json = REPO_ROOT / "weights" / "runs" / "p2_bakeoff" / "bakeoff_summary.json"

if bakeoff_md.exists():
    print(bakeoff_md.read_text())
else:
    print(f"Bake-off table not found at {bakeoff_md}; did the previous cell finish?")

print("\n--- Machine-readable summary ---")
if bakeoff_json.exists():
    with bakeoff_json.open() as f:
        data = json.load(f)
    for r in sorted(data.get("ran", []), key=lambda x: -(x.get("map5095") or 0)):
        print(f"  {r['name']:<28} mAP@[.50:.95]={r.get('map5095')}  status={r['status']}")
else:
    print(f"Summary JSON not found at {bakeoff_json}")

# Section 6 — EXP 2.3.1: Promote top 2 winners to full 50-epoch training (~5 hr, ~$3)

**Goal:** for each of the top 2 inits from Section 5, run full 50-epoch training, then submit each to Kaggle.

After this section finishes, you have:
- 1 anchor submission (Section 4)
- 2 full-training submissions (this section)
- A clear PRIMARY init (best LB) and SECONDARY init (runner-up) for Phase 3-7

The PRIMARY init is the one we invest the heaviest in for Phases 3-7. The SECONDARY runs in parallel during Phase 6's 5-fold ensemble for architectural diversity.

The cell below auto-detects the top 2 from the bake-off summary JSON and trains both.

In [ ]:
"""EXP 2.3.1 -- promote top 2 inits to full 50-epoch training."""
from scripts.eval_pretrained_inits import CANDIDATES, _resolve_init_arg

if not bakeoff_json.exists():
    raise SystemExit("Run Section 5 first")
with bakeoff_json.open() as f:
    bakeoff = json.load(f)
ranked = sorted(
    [r for r in bakeoff.get("ran", []) if r["status"] == "OK" and r.get("map5095") is not None],
    key=lambda x: -x["map5095"],
)
print(f"Top 3 from bake-off:")
for i, r in enumerate(ranked[:3], 1):
    print(f"  {i}. {r['name']:<28} val mAP@[.50:.95] = {r['map5095']:.4f}")

top1 = ranked[0]
top2 = ranked[1] if len(ranked) > 1 else None
gap = (top1["map5095"] - top2["map5095"]) if top2 else 0
promote = [top1]
if top2 and gap < 0.05:
    promote.append(top2)
    if len(ranked) > 2 and gap < 0.02:
        promote.append(ranked[2])
print(f"\nPromoting {len(promote)} init(s) to full training (gap = {gap:+.4f}):")
for r in promote:
    print(f"  - {r['name']}")

In [ ]:
"""Train each promoted init for the full 50 epochs (~2 hr each on RTX 4090)."""
full_results = []
for r in promote:
    candidate = next((c for c in CANDIDATES if c.name == r["name"]), None)
    if candidate is None:
        print(f"[skip] {r['name']} not in candidate registry")
        continue
    init_arg = _resolve_init_arg(candidate)
    cfg = TrainConfig(
        init=init_arg,
        epochs=50,
        imgsz=640,
        fold=0,
        batch=16,
        name=f"p2_full_{candidate.name}",
        extra_tags=["full", "promoted_from_bakeoff"],
    )
    print(f"\n>>> Training {candidate.name} (full 50 epochs) <<<")
    res = train_one_run(cfg)
    full_results.append((candidate.name, res))

In [ ]:
"""Generate Kaggle CSVs for each promoted init and print submit commands."""
submit_cmds = []
for init_name, res in full_results:
    weights = res["best_pt"]
    out_csv = REPO_ROOT / "submissions" / f"p2_full_{init_name}.csv"
    cmd_predict = [
        sys.executable, str(REPO_ROOT / "scripts" / "predict_test_set.py"),
        "--weights", str(weights),
        "--out", str(out_csv),
        "--imgsz", "640",
    ]
    print(" ".join(cmd_predict))
    subprocess.check_call(cmd_predict, cwd=str(REPO_ROOT))
    submit_cmds.append(
        f"kaggle competitions submit -c fathomnet-2026 -f {out_csv} "
        f"-m 'Phase 2 full training: {init_name}'"
    )

print("\n\n=== SUBMIT THESE TO KAGGLE (uses 2 daily slots) ===")
for c in submit_cmds:
    print(c)

# Section 7 — Results summary (FILL IN AFTER EACH SUBMISSION)

After each Kaggle submission lands, paste the LB score into the table below. This becomes the source data for `notes/phase2_results.md` and the working notes paper.

## Submission log

| Run | val mAP@[.50:.95] | Kaggle LB mAP | Date | Notes |
|---|---|---|---|---|
| EXP 2.2 vanilla yolo11m+COCO | _fill_ | _fill_ | _date_ | First submission, anchors LB |
| EXP 2.3.1 PRIMARY (`<init_name>`) | _fill_ | _fill_ | _date_ | Bake-off winner, full 50ep |
| EXP 2.3.1 SECONDARY (`<init_name>`) | _fill_ | _fill_ | _date_ | Runner-up, full 50ep |

## Per-class AP comparison (advanced — fill from W&B)

W&B logs per-class AP under `metrics/AP_class_<idx>`. Compare PRIMARY vs SECONDARY for each class — divergent strengths inform per-class NMS tuning in Phase 7 EXP 7.6.

| Class | Vanilla AP | PRIMARY AP | SECONDARY AP | PRIMARY > SECONDARY? |
|---|---|---|---|---|
| (32 rows -- fill from W&B) |  |  |  |  |

## Decision: which init becomes PRIMARY for Phase 3-7?

**PRIMARY:** `<init_name>` — chosen because: _fill in reasoning_

**SECONDARY (Phase 6 ensemble member):** `<init_name>` — chosen because: _fill in reasoning_

**Inits dropped from further investment:** `<list>` — reason: _fill in_

## Next: Phase 3 (Block E in master_checklist.txt)

After Phase 2 wraps:

1. Stop the RunPod pod (cost discipline R5)
2. Commit `notes/phase2_results.md` + this notebook with the filled-in table
3. Read master_checklist.txt Block E for Phase 3 PU + class imbalance experiments
4. Phase 3 starts from PRIMARY init's `best.pt` weights